# Assignment 4: Sports Team Performance vs. City Population

## Overview
This assignment investigates whether larger metropolitan areas tend to have more successful sports teams. We analyze 2018 win/loss data from all four major North American sports leagues (NHL, NBA, MLB, NFL) and compute their Pearson correlation with the metropolitan population.

## Learning Objectives
- Parse HTML tables with `pd.read_html()`
- Clean team names with regex to extract and map city/area identifiers
- Use `groupby` to aggregate multiple teams in the same metropolitan area
- Compute Pearson correlation with `scipy.stats.pearsonr`
- Run paired t-tests with `scipy.stats.ttest_rel` across multiple groups

## Data Files
| File | Description |
|------|-------------|
| `assets/wikipedia_data.html` | Metropolitan areas with population and team names |
| `assets/nhl.csv` | NHL season records by year |
| `assets/nba.csv` | NBA season records by year |
| `assets/mlb.csv` | MLB season records by year |
| `assets/nfl.csv` | NFL season records by year |

## Key Strategy
For each sport:
1. Filter `cities` to the sport team column; drop rows with `—` or empty
2. Filter sport CSV to year 2018; drop division header rows
3. Strip asterisks from team names (playoff-qualified teams have `*`)
4. Map each team to its metropolitan area via string matching on the team's last word
5. Compute win/loss ratio = W / (W + L); average for multi-team cities
6. Merge with population data and compute Pearson correlation

## Notes
- Metropolitan areas may have multiple teams in a single sport — average their win/loss ratios
- Strip footnotes `[note N]` and asterisks `*` from the Wikipedia HTML table using regex

## Question 1
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NHL** using **2018** data.

### Approach
Clean the Wikipedia cities table by stripping footnotes and asterisks. Index by the NHL team name column. Filter nhl.csv to 2018, drop division header rows, strip asterisks from team names (e.g., "Boston Bruins*" -> "Boston Bruins"), then match each team's last word against the cities index. Group by metropolitan area, compute W/(W+L), merge with population, return pearsonr.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

nhl_df = pd.read_csv("assets/nhl.csv")
cities = pd.read_html("assets/wikipedia_data.html")[1]
cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]

def nhl_correlation():
    cities_nhl = cities.copy()
    for col in cities_nhl.columns:
        cities_nhl[col] = cities_nhl[col].str.replace(r'\[note\s?\d*\]', '', regex=True)
        cities_nhl[col] = cities_nhl[col].str.replace(r'[*+]', '', regex=True)
    cities_nhl.rename(columns={"Population (2016 est.)[8]": "Population"}, inplace=True)
    cities_nhl = cities_nhl[['Metropolitan area', 'NHL', 'Population']]
    cities_nhl = cities_nhl[~cities_nhl['NHL'].isin(['—', ''])]
    cities_nhl['Population'] = pd.to_numeric(cities_nhl['Population'], errors='coerce')
    cities_nhl = cities_nhl.sort_values('Metropolitan area').set_index('NHL')

    nhl_2018 = nhl_df[nhl_df['year'] == 2018].copy()
    divisions = ['Atlantic Division', 'Metropolitan Division', 'Central Division', 'Pacific Division']
    nhl_2018 = nhl_2018[~nhl_2018['team'].isin(divisions)]
    nhl_2018['team'] = nhl_2018['team'].str.replace(r'[*+]', '', regex=True).str.strip()
    nhl_2018['W'] = pd.to_numeric(nhl_2018['W'], errors='coerce')
    nhl_2018['L'] = pd.to_numeric(nhl_2018['L'], errors='coerce')

    def find_area(team_name):
        token = team_name.split()[-1]
        for key in cities_nhl.index:
            if token in key:
                return cities_nhl.at[key, 'Metropolitan area']
        return None

    nhl_2018['area'] = nhl_2018['team'].apply(find_area)
    nhl_2018 = nhl_2018.dropna(subset=['area'])
    grouped = nhl_2018.groupby('area').agg(W=('W', 'sum'), L=('L', 'sum'))
    grouped['win_loss'] = grouped['W'] / (grouped['W'] + grouped['L'])

    pop = cities_nhl[['Population', 'Metropolitan area']].reset_index(drop=True).drop_duplicates('Metropolitan area').set_index('Metropolitan area')
    merged = grouped.join(pop, how='inner')

    population_by_region = merged['Population'].values.astype(float)
    win_loss_by_region = merged['win_loss'].values

    assert len(population_by_region) == len(win_loss_by_region), "Q1: Your lists must be the same length"
    assert len(population_by_region) == 28, "Q1: There should be 28 teams being analysed for NHL"
    return stats.pearsonr(population_by_region, win_loss_by_region)

nhl_correlation()

PearsonRResult(statistic=0.012308996455744264, pvalue=0.9504308637909508)

## Question 2
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NBA** using **2018** data.

### Approach
Same pattern as NHL. Additionally strip playoff seed numbers in parentheses (e.g., "Boston Celtics (1)" -> "Boston Celtics") from team names before matching.

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

nba_df = pd.read_csv("assets/nba.csv")
cities = pd.read_html("assets/wikipedia_data.html")[1]
cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]

def nba_correlation():
    cities_nba = cities.copy()
    for col in cities_nba.columns:
        cities_nba[col] = cities_nba[col].str.replace(r'\[note\s?\d*\]', '', regex=True)
        cities_nba[col] = cities_nba[col].str.replace(r'[*+]', '', regex=True)
    cities_nba.rename(columns={"Population (2016 est.)[8]": "Population"}, inplace=True)
    cities_nba = cities_nba[['Metropolitan area', 'NBA', 'Population']]
    cities_nba = cities_nba[~cities_nba['NBA'].isin(['—', ''])]
    cities_nba['Population'] = pd.to_numeric(cities_nba['Population'], errors='coerce')
    cities_nba = cities_nba.sort_values('Metropolitan area').set_index('NBA')

    nba_2018 = nba_df[nba_df['year'] == 2018].copy()
    divisions = ['Atlantic Division', 'Central Division', 'Southeast Division',
                 'Northwest Division', 'Pacific Division', 'Southwest Division']
    nba_2018 = nba_2018[~nba_2018['team'].isin(divisions)]
    nba_2018['team'] = nba_2018['team'].str.replace(r'[*+]', '', regex=True).str.strip()
    nba_2018['team'] = nba_2018['team'].str.replace(r'\s*\(\d+\)', '', regex=True).str.strip()
    nba_2018['W'] = pd.to_numeric(nba_2018['W'], errors='coerce')
    nba_2018['L'] = pd.to_numeric(nba_2018['L'], errors='coerce')

    def find_area(team_name):
        token = team_name.split()[-1]
        for key in cities_nba.index:
            if token in key:
                return cities_nba.at[key, 'Metropolitan area']
        return None

    nba_2018['area'] = nba_2018['team'].apply(find_area)
    nba_2018 = nba_2018.dropna(subset=['area'])
    grouped = nba_2018.groupby('area').agg(W=('W', 'sum'), L=('L', 'sum'))
    grouped['win_loss'] = grouped['W'] / (grouped['W'] + grouped['L'])

    pop = cities_nba.reset_index()[['Population', 'Metropolitan area']].drop_duplicates('Metropolitan area').set_index('Metropolitan area')
    merged = grouped.join(pop, how='inner')

    population_by_region = merged['Population'].values.astype(float)
    win_loss_by_region = merged['win_loss'].values

    assert len(population_by_region) == len(win_loss_by_region), "Q2: Your lists must be the same length"
    assert len(population_by_region) == 28, "Q2: There should be 28 teams being analysed for NBA"
    return stats.pearsonr(population_by_region, win_loss_by_region)

nba_correlation()

PearsonRResult(statistic=-0.17657160252844611, pvalue=0.36874741604462974)

## Question 3
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **MLB** using **2018** data.

### Approach
MLB provides a pre-computed `W-L%` column (win percentage). Use that directly instead of computing from W and L. Average W-L% per metropolitan area. Manual fix: "Chicago White Sox" last word is "Sox", which would match "Red Sox" (Boston) — override area to "Chicago".

In [3]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

mlb_df = pd.read_csv("assets/mlb.csv")
cities = pd.read_html("assets/wikipedia_data.html")[1]
cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]

def mlb_correlation():
    cities_mlb = cities.copy()
    for col in cities_mlb.columns:
        cities_mlb[col] = cities_mlb[col].str.replace(r'\[note\s?\d*\]', '', regex=True)
        cities_mlb[col] = cities_mlb[col].str.replace(r'[*+]', '', regex=True)
    cities_mlb.rename(columns={"Population (2016 est.)[8]": "Population"}, inplace=True)
    cities_mlb = cities_mlb[['Metropolitan area', 'MLB', 'Population']]
    cities_mlb = cities_mlb[~cities_mlb['MLB'].isin(['—', ''])]
    cities_mlb['Population'] = pd.to_numeric(cities_mlb['Population'], errors='coerce')
    cities_mlb = cities_mlb.sort_values('Metropolitan area').set_index('MLB')

    mlb_2018 = mlb_df[mlb_df['year'] == 2018].copy()
    mlb_2018 = mlb_2018[~mlb_2018['team'].str.contains('Division', na=False)]
    mlb_2018['team'] = mlb_2018['team'].str.replace(r'[*+]', '', regex=True).str.strip()
    mlb_2018['W-L%'] = pd.to_numeric(mlb_2018['W-L%'], errors='coerce')

    def find_area(team_name):
        token = team_name.split()[-1]
        for key in cities_mlb.index:
            if token in key:
                return cities_mlb.at[key, 'Metropolitan area']
        return None

    mlb_2018['area'] = mlb_2018['team'].apply(find_area)
    mlb_2018.loc[mlb_2018['team'] == 'Chicago White Sox', 'area'] = 'Chicago'
    mlb_2018 = mlb_2018.dropna(subset=['area'])

    grouped = mlb_2018.groupby('area')['W-L%'].mean().reset_index()
    grouped.columns = ['area', 'win_loss']
    grouped = grouped.set_index('area')

    pop = cities_mlb.reset_index()[['Population', 'Metropolitan area']].drop_duplicates('Metropolitan area').set_index('Metropolitan area')
    merged = grouped.join(pop, how='inner')

    population_by_region = merged['Population'].values.astype(float)
    win_loss_by_region = merged['win_loss'].values

    assert len(population_by_region) == len(win_loss_by_region), "Q3: Your lists must be the same length"
    assert len(population_by_region) == 26, "Q3: There should be 26 teams being analysed for MLB"
    return stats.pearsonr(population_by_region, win_loss_by_region)

mlb_correlation()

PearsonRResult(statistic=0.15003737475409495, pvalue=0.46442827201123316)

## Question 4
For this question, calculate the win/loss ratio's correlation with the population of the city it is in for the **NFL** using **2018** data.

### Approach
Same pattern as NHL. Drop 8 AFC/NFC division header rows. The Buffalo Bills have Toronto listed as a metropolitan area in the cities HTML — drop Toronto from the NFL cities to avoid double-counting. Strip asterisks before team-name matching.

In [4]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

nfl_df = pd.read_csv("assets/nfl.csv")
cities = pd.read_html("assets/wikipedia_data.html")[1]
cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]

def nfl_correlation():
    cities_nfl = cities.copy()
    for col in cities_nfl.columns:
        cities_nfl[col] = cities_nfl[col].str.replace(r'\[note\s?\d*\]', '', regex=True)
        cities_nfl[col] = cities_nfl[col].str.replace(r'[*+]', '', regex=True)
    cities_nfl.rename(columns={"Population (2016 est.)[8]": "Population"}, inplace=True)
    cities_nfl = cities_nfl[['Metropolitan area', 'NFL', 'Population']]
    cities_nfl = cities_nfl[~cities_nfl['NFL'].isin(['—', ''])]
    cities_nfl = cities_nfl[cities_nfl['Metropolitan area'] != 'Toronto']
    cities_nfl['Population'] = pd.to_numeric(cities_nfl['Population'], errors='coerce')
    cities_nfl = cities_nfl.sort_values('Metropolitan area').set_index('NFL')

    nfl_2018 = nfl_df[nfl_df['year'] == 2018].copy()
    divisions = ['AFC East', 'AFC North', 'AFC South', 'AFC West',
                 'NFC East', 'NFC North', 'NFC South', 'NFC West']
    nfl_2018 = nfl_2018[~nfl_2018['team'].isin(divisions)]
    nfl_2018['team'] = nfl_2018['team'].str.replace(r'[*+]', '', regex=True).str.strip()
    nfl_2018['W'] = pd.to_numeric(nfl_2018['W'], errors='coerce')
    nfl_2018['L'] = pd.to_numeric(nfl_2018['L'], errors='coerce')

    def find_area(team_name):
        token = team_name.split()[-1]
        for key in cities_nfl.index:
            if token in key:
                return cities_nfl.at[key, 'Metropolitan area']
        return None

    nfl_2018['area'] = nfl_2018['team'].apply(find_area)
    nfl_2018 = nfl_2018.dropna(subset=['area'])
    grouped = nfl_2018.groupby('area').agg(W=('W', 'sum'), L=('L', 'sum'))
    grouped['win_loss'] = grouped['W'] / (grouped['W'] + grouped['L'])

    pop = cities_nfl.reset_index()[['Population', 'Metropolitan area']].drop_duplicates('Metropolitan area').set_index('Metropolitan area')
    merged = grouped.join(pop, how='inner')

    population_by_region = merged['Population'].values.astype(float)
    win_loss_by_region = merged['win_loss'].values

    assert len(population_by_region) == len(win_loss_by_region), "Q4: Your lists must be the same length"
    assert len(population_by_region) == 29, "Q4: There should be 29 teams being analysed for NFL"
    return stats.pearsonr(population_by_region, win_loss_by_region)

nfl_correlation()

PearsonRResult(statistic=0.004922112149349456, pvalue=0.9797833458363692)

## Question 5
In this question I would like you to explore the hypothesis that **given that an area has two sports teams in different sports, those teams will perform the same within their respective sports**. How I would like to see this explored is with a series of paired t-tests (so use [`ttest_rel`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)) between all pairs of sports. Are there any sports where we can reject the null hypothesis? Again, average values where a sport has multiple teams in one region. Remember, you will only be including, for each sport, cities which have teams engaged in that sport, drop others as appropriate.

### Approach
Build a win/loss ratio Series for each of the 4 sports (reusing the same team-mapping logic). For each pair of sports, find cities that have teams in BOTH sports (inner join on metropolitan area). Run `ttest_rel` on the paired win/loss ratios for those shared cities. Build a symmetric 4x4 p-value DataFrame with NaN on the diagonal.

In [5]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import re

mlb_df = pd.read_csv("assets/mlb.csv")
nhl_df = pd.read_csv("assets/nhl.csv")
nba_df = pd.read_csv("assets/nba.csv")
nfl_df = pd.read_csv("assets/nfl.csv")
cities = pd.read_html("assets/wikipedia_data.html")[1]
cities = cities.iloc[:-1, [0, 3, 5, 6, 7, 8]]

def get_sport_win_loss(sport_csv, sport_col, division_keywords=None):
    cities_s = cities.copy()
    for col in cities_s.columns:
        cities_s[col] = cities_s[col].str.replace(r'\[note\s?\d*\]', '', regex=True)
        cities_s[col] = cities_s[col].str.replace(r'[*+]', '', regex=True)
    cities_s.rename(columns={"Population (2016 est.)[8]": "Population"}, inplace=True)
    cities_s = cities_s[['Metropolitan area', sport_col, 'Population']]
    cities_s = cities_s[~cities_s[sport_col].isin(['—', ''])]
    if sport_col == 'NFL':
        cities_s = cities_s[cities_s['Metropolitan area'] != 'Toronto']
    cities_s = cities_s.set_index(sport_col)

    df_2018 = sport_csv[sport_csv['year'] == 2018].copy()
    if division_keywords:
        df_2018 = df_2018[~df_2018['team'].isin(division_keywords)]
    df_2018['team'] = df_2018['team'].str.replace(r'[*+]', '', regex=True).str.strip()
    if sport_col == 'NBA':
        df_2018['team'] = df_2018['team'].str.replace(r'\s*\(\d+\)', '', regex=True).str.strip()

    def find_area(team_name):
        token = team_name.split()[-1]
        for key in cities_s.index:
            if token in key:
                return cities_s.at[key, 'Metropolitan area']
        return None

    df_2018['area'] = df_2018['team'].apply(find_area)
    if sport_col == 'MLB':
        df_2018.loc[df_2018['team'] == 'Chicago White Sox', 'area'] = 'Chicago'
        df_2018['W-L%'] = pd.to_numeric(df_2018['W-L%'], errors='coerce')
        df_2018 = df_2018.dropna(subset=['area'])
        return df_2018.groupby('area')['W-L%'].mean()
    else:
        df_2018['W'] = pd.to_numeric(df_2018['W'], errors='coerce')
        df_2018['L'] = pd.to_numeric(df_2018['L'], errors='coerce')
        df_2018 = df_2018.dropna(subset=['area'])
        grouped = df_2018.groupby('area').agg(W=('W', 'sum'), L=('L', 'sum'))
        grouped['win_loss'] = grouped['W'] / (grouped['W'] + grouped['L'])
        return grouped['win_loss']

def sports_team_performance():
    nhl_wl = get_sport_win_loss(nhl_df, 'NHL',
        ['Atlantic Division', 'Metropolitan Division', 'Central Division', 'Pacific Division'])
    nba_wl = get_sport_win_loss(nba_df, 'NBA',
        ['Atlantic Division', 'Central Division', 'Southeast Division',
         'Northwest Division', 'Pacific Division', 'Southwest Division'])
    mlb_wl = get_sport_win_loss(mlb_df, 'MLB', None)
    nfl_wl = get_sport_win_loss(nfl_df, 'NFL',
        ['AFC East', 'AFC North', 'AFC South', 'AFC West',
         'NFC East', 'NFC North', 'NFC South', 'NFC West'])

    sport_data = {'NFL': nfl_wl, 'NBA': nba_wl, 'NHL': nhl_wl, 'MLB': mlb_wl}
    sports = ['NFL', 'NBA', 'NHL', 'MLB']
    p_values = pd.DataFrame({k: np.nan for k in sports}, index=sports)

    for s1 in sports:
        for s2 in sports:
            if s1 != s2:
                common = pd.merge(sport_data[s1].rename('s1'), sport_data[s2].rename('s2'),
                                  left_index=True, right_index=True, how='inner').dropna()
                _, p = stats.ttest_rel(common['s1'], common['s2'])
                p_values.loc[s1, s2] = p

    assert abs(p_values.loc["NBA", "NHL"] - 0.02) <= 0.05, "The NBA-NHL p-value should be around 0.02"
    assert abs(p_values.loc["MLB", "NFL"] - 0.80) <= 0.10, "The MLB-NFL p-value should be around 0.80"
    return p_values

sports_team_performance()

,NFL,NBA,NHL,MLB
NFL,NaN,0.941792,0.030959,0.867748
NBA,0.941792,NaN,0.022316,0.971079
NHL,0.030959,0.022316,NaN,0.002552
MLB,0.867748,0.971079,0.002552,NaN
